# Predictive Analysis in Diabetes

Production-quality version of the original *Hands-On Predictive Analysis in Diabetes* notebook. The original code and its intent are preserved throughout; this version adds clearer structure, richer documentation, centralised configuration, and a small number of fixes required to make the code run reliably.

## Dataset

The Pima Indians Diabetes dataset (`diabetes.csv`, 768 records, 8 features, 1 binary target):

| Column | Description |
| --- | --- |
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration (oral glucose tolerance test) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-hour serum insulin (mu U/ml) |
| BMI | Body mass index (kg/m^2) |
| DiabetesPedigreeFunction | Genetic predisposition score |
| Age | Age in years |
| **Outcome** | **Target:** 1 = diabetic, 0 = non-diabetic |

## Pipeline Overview

1. Load and inspect the data
2. Exploratory analysis: correlations, descriptive statistics, feature distributions
3. Impute physiologically impossible zero values (mean for symmetric features, median for skewed ones)
4. Detect and remove outliers (IQR box-plot rule, then a 95th-percentile trim on Insulin)
5. Standardise the features with `StandardScaler`
6. Train/test split, then SMOTE oversampling of the minority class (training data only)
7. Train and evaluate three classifiers: Logistic Regression, K-Nearest Neighbors, Gaussian Naive Bayes
8. Persist the fitted model with `pickle`

## Changes vs. the Original Notebook

- **Bug fix (outlier loop):** the original loop referenced undefined variables (`X_outlier_detection_total` / `y_outlier_detection_total`) and would raise a `NameError`. It now updates `X_outlier_detection` / `y_outlier_detection` correctly.
- **Deprecated API:** `sns.distplot` is deprecated and removed in recent seaborn releases. It is replaced by `sns.histplot(..., kde=True)`, which conveys the same histogram + density information.
- **Configuration:** the data path, test size and random seed are defined once in Section 2 instead of being hard-coded in each cell.
- Everything else (logic, order of operations, models, metrics and outputs) follows the original notebook.

## Requirements

```bash
pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn
```


## 1. Setup and Imports

All required libraries are imported once, up front, as is conventional for production code. The scikit-learn / imbalanced-learn imports stay inside their sections so each modelling step remains self-contained, mirroring the original notebook.


In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## 2. Configuration

A single place for paths and reproducibility settings. Every downstream cell reads these constants, so the notebook can be retargeted (e.g. to a different data location or split ratio) by editing one cell.


In [ ]:
from pathlib import Path

# Location of the dataset. Adjust if running elsewhere, e.g. Path("/content/diabetes.csv") in Google Colab.
DATA_PATH = Path("diabetes.csv")

# Single seed reused by the train/test split and SMOTE so results are reproducible.
RANDOM_STATE = 42

# Fraction of the data held back for the final model evaluation.
TEST_SIZE = 0.33


## 3. Load and Inspect the Dataset

Load the CSV into a pandas DataFrame and take a first look at the records.


In [ ]:
dataframe = pd.read_csv(DATA_PATH)
dataframe.head()


### 3.1 Columns and Data Types

Confirm the expected schema: eight numeric features plus the integer `Outcome` target. All columns are numeric, so no categorical encoding is required.


In [ ]:
print("Columns:")
print(dataframe.columns.tolist())

print("\nData types:")
print(dataframe.dtypes)


### 3.2 Missing Values

`isnull().sum()` counts true `NaN` values per column. This dataset reports **no `NaN`s**, but inspecting the first 20 rows reveals a different problem: several clinical measurements are recorded as `0` (e.g. blood pressure, insulin, BMI), which is physiologically impossible. Those zeros are hidden missing values and are imputed in Section 5.


In [ ]:
dataframe.isnull().sum()


In [ ]:
dataframe.head(20)


### 3.3 Dataset Size and Task Framing

- **Target column:** `Outcome` with values [0, 1], so this is a **binary classification** task.
- Because the target column is available in the dataset, this is a **supervised** machine learning problem.
- **Records:** 768.


In [ ]:
dataframe.shape


## 4. Exploratory Data Analysis

### 4.1 Correlation Coefficient

The Pearson correlation matrix quantifies the linear relationship between each pair of numeric columns. Values near +1/-1 indicate strong positive/negative relationships; values near 0 indicate little linear association. Features that correlate strongly with `Outcome` (notably `Glucose`) are expected to be the most predictive. The heatmap is also saved to `correlation-coefficient.jpg` for reporting.


In [ ]:
dataframe.corr()


In [ ]:
plt.figure(figsize=(15, 15))
ax = sns.heatmap(dataframe.corr(), annot=True)
plt.savefig("correlation-coefficient.jpg")
plt.show()


### 4.2 Descriptive Statistics

`describe()` reports count, mean, standard deviation, minimum, quartiles and maximum for every column. It is useful for:

- **Spotting invalid values**, e.g. the zeros that stand in for missing measurements (a minimum blood pressure of 0 is not a real observation).
- **Comparing mean vs. median** (the 50% quartile): when the mean is pulled well above the median, the distribution is right-skewed and the median is the safer imputation statistic.
- **Seeing the scale differences between features**, which motivates standardisation in Section 7.


In [ ]:
dataframe.describe()


### 4.3 Feature Distributions and Imputation Strategy

The shape of each distribution decides how its zero values should be imputed:

- **Approximately symmetric distribution → mean** (uses every observation, no distortion).
- **Skewed distribution → median** (robust to the outliers that cause the skew).
- **Categorical data → mode** (not needed here; all columns are numeric).

> Note: the original notebook used `sns.distplot`, which is deprecated and has been removed in recent seaborn versions. The equivalent modern call is `sns.histplot(..., kde=True)`.

| Feature | Imputation statistic | Distribution | Rationale |
| --- | --- | --- | --- |
| Pregnancies | Median | Right-skewed | Count data with a long right tail; median resists extreme values |
| Glucose | Mean | Roughly symmetric | Mean is representative when the distribution is balanced |
| BloodPressure | Mean | Roughly symmetric | Mean is representative when the distribution is balanced |
| SkinThickness | Median | Right-skewed | Median resists extreme values |
| Insulin | Median | Strongly right-skewed | Heavy tail with extreme values; median is far more robust |
| BMI | Mean | Roughly symmetric | Mean is representative when the distribution is balanced |
| DiabetesPedigreeFunction | Median | Right-skewed | Median resists extreme values |
| Age | Median | Right-skewed | Median resists extreme values |


In [ ]:
feature_columns = [col for col in dataframe.columns if col != "Outcome"]

for col in feature_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframe[col], kde=True)
    plt.title(f"Distribution of {col} (before imputation)")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


## 5. Data Imputation of Zero Values

Each feature's zero entries are replaced by the statistic chosen in Section 4.3. The replacements follow the original notebook exactly, including the order of operations (Insulin first, then the remaining columns).

> Caveat kept from the original: each statistic is computed on the raw column, i.e. while the zeros are still present. A slightly more robust variant would compute the mean/median over the non-zero values only.


In [ ]:
# Insulin -> strongly right-skewed distribution -> median
dataframe["Insulin"] = dataframe["Insulin"].replace(0, dataframe["Insulin"].median())

dataframe["Pregnancies"] = dataframe["Pregnancies"].replace(0, dataframe["Pregnancies"].median())
dataframe["Glucose"] = dataframe["Glucose"].replace(0, dataframe["Glucose"].mean())
dataframe["BloodPressure"] = dataframe["BloodPressure"].replace(0, dataframe["BloodPressure"].mean())
dataframe["SkinThickness"] = dataframe["SkinThickness"].replace(0, dataframe["SkinThickness"].median())
dataframe["BMI"] = dataframe["BMI"].replace(0, dataframe["BMI"].mean())
dataframe["DiabetesPedigreeFunction"] = dataframe["DiabetesPedigreeFunction"].replace(0, dataframe["DiabetesPedigreeFunction"].median())
dataframe["Age"] = dataframe["Age"].replace(0, dataframe["Age"].median())

dataframe.head(20)


### 5.1 Section Recap

- **Descriptive statistics** and their significance: summary measures that reveal invalid values, skew and scale differences.
- **Correlation coefficient** and its significance: strength and direction of linear relationships between features and the target.
- **Types of distributions** and their significance: the shape of a feature decides whether the mean or the median represents it best.
- **Why the median is more robust to outliers:** it depends only on the middle rank of the sorted values, so extreme observations shift it very little, whereas the mean is dragged toward every extreme value.
- **Imputation rules:** numeric + symmetric → mean; numeric + skewed → median; categorical → mode.


## 6. Outlier Detection and Removal (IQR Method)

Separate the input features `X` from the target `y`, then visualise each feature with a box plot. Points beyond the whiskers are candidate outliers.


In [ ]:
# X -> input features, y -> target value
X = dataframe.drop(columns="Outcome", axis=1)
y = dataframe["Outcome"]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
sns.boxplot(data=X, ax=ax)
plt.savefig("boxPlot.jpg")
plt.show()


### 6.1 IQR-Based Row Filtering

For every feature, compute the interquartile range `IQR = Q3 - Q1` and keep only the rows whose value lies within the fences:

- lower bound = `Q1 - 1.5 * IQR`
- upper bound = `Q3 + 1.5 * IQR`

The filter is applied feature by feature, and the surviving rows are carried forward in `X_outlier_detection` / `y_outlier_detection` so the features and labels always stay aligned.

> **Bug fix vs. the original:** the original loop assigned to `X_outlier_detection_total` / `y_outlier_detection_total`, which were never defined, so it raised a `NameError`. The loop below performs the intended cumulative filtering.


In [ ]:
cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

X_outlier_detection = X.copy()
y_outlier_detection = y.copy()

for col in cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    mask = (X[col] >= lower_bound) & (X[col] <= upper_bound)

    X_outlier_detection = X_outlier_detection[mask]
    y_outlier_detection = y_outlier_detection[mask]

    print(f"{col}: kept {mask.sum()} of {len(mask)} rows")

print("\nFinal shape after IQR filtering:")
print("X:", X_outlier_detection.shape)
print("y:", y_outlier_detection.shape)


## 7. Standardization

Transform every feature to the **standard normal form**: mean = 0 and standard deviation = 1, using scikit-learn's `StandardScaler`.

This matters because the raw features live on very different scales (Insulin in the hundreds, DiabetesPedigreeFunction below 2.5). Without scaling, models that rely on distances or gradients (KNN, Logistic Regression with regularisation) are biased toward the large-scale features. Standardisation puts every feature on equal footing and reduces that bias in the model.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_outlier_detection)

X_scaled[:5]


Box plots of the standardised features: every column is now centred on 0 with a comparable spread, which makes the remaining outliers directly comparable across features.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
sns.boxplot(data=X_scaled, ax=ax)
plt.savefig("boxPlot_scaled.jpg")
plt.show()


`fit_transform` returns a NumPy array. Converting it back to a DataFrame restores the column names, so the feature names survive into model inspection and later analysis.


In [ ]:
print(type(X_scaled))

X_scaled = pd.DataFrame(X_scaled, columns=cols)
X_scaled.describe()


## 8. Approach 2: Quantile-Based Trimming (Insulin)

The IQR pass removed the most extreme values, but the box plot of the scaled data shows `Insulin` still carries a heavy upper tail. As a second, coarser pass, all rows whose standardised `Insulin` exceeds its **95th percentile** are dropped. This trims the top 5% of Insulin values in one step.

The index is reset first so the feature rows and the target rows stay aligned after filtering.


In [ ]:
X_scaled.reset_index(drop=True, inplace=True)
y_outlier_detection.reset_index(drop=True, inplace=True)

q = X_scaled["Insulin"].quantile(0.95)
mask = X_scaled["Insulin"] < q

dataNew = X_scaled[mask]
y_outlier_detection = y_outlier_detection[mask]

print("X shape after 95th-percentile trim:", dataNew.shape)
print("y shape after 95th-percentile trim:", y_outlier_detection.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
sns.boxplot(data=dataNew, ax=ax)
plt.savefig("boxPlot_trimmed.jpg")
plt.show()


## 9. Model Training Preparation: Train/Test Split

Hold out 33% of the data for the final evaluation and train on the remaining 67%. The split uses the shared `RANDOM_STATE`, so running the notebook twice produces identical splits.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataNew,
    y_outlier_detection,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)


## 10. Handling Class Imbalance

The outcome classes are imbalanced (many more non-diabetic than diabetic patients). If ignored, a model can score well on accuracy while almost never predicting the minority class. Three standard remedies:

- **Oversampling:** duplicate/randomly repeat samples of the minority class until it matches the majority class.
- **Undersampling:** discard samples of the majority class until it matches the minority class.
- **SMOTE (Synthetic Minority Over-sampling Technique):** create *synthetic* minority samples by interpolating between neighbouring minority observations, increasing the minority class to the size of the majority class without exact duplicates.

First, check the imbalance in the training data.


In [ ]:
y_train.value_counts()


### 10.1 Apply SMOTE

SMOTE is fitted **on the training data only**. The test set must stay untouched so it remains an honest estimate of real-world performance; resampling it (or resampling before the split) would leak information into the evaluation.


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Check the resampled class distribution: both classes should now be balanced.
print("Resampled class distribution:")
print(pd.Series(y_train_resampled).value_counts())


## 11. Model 1: Logistic Regression

A linear baseline classifier: fast, interpretable coefficients, and a strong reference point for this kind of tabular clinical data.

> **Healthcare note:** recall is the most important metric here. A false negative (a diabetic patient predicted as healthy) can delay treatment, which is usually costlier than a false positive that leads to a follow-up test.


In [ ]:
from sklearn.linear_model import LogisticRegression

classification = LogisticRegression()
classification.fit(X_train_resampled, y_train_resampled)


### 11.1 Model Predictions


In [ ]:
y_predictions = classification.predict(X_test)
print(y_predictions)


### 11.2 Model Evaluation


In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_predictions)


The classification report breaks accuracy down per class. For the `Diabetic` class, focus on **recall** (share of true diabetic patients correctly detected) and its trade-off with precision.


In [ ]:
from sklearn.metrics import classification_report

target_names = ["Non-Diabetic", "Diabetic"]
print(classification_report(y_test, y_predictions, target_names=target_names))


## 12. Model Persistence

Serialize the fitted model with `pickle` so it can be deployed or reloaded later without retraining, then verify the round trip by reloading it and scoring the test set.


In [ ]:
pickle.dump(classification, open("classification_model.pkl", "wb"))

classification_model = pickle.load(open("classification_model.pkl", "rb"))
classification_model.predict(X_test)


## 13. Model 2: K-Nearest Neighbors Classifier

KNN classifies a patient by majority vote among the `k` closest training samples. It is sensitive to feature scale (already handled by standardisation in Section 7) and to class imbalance (already handled by SMOTE in Section 10).


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

knn = KNeighborsClassifier()
knn.fit(X_train_resampled, y_train_resampled)


### 13.1 Model Predictions and Evaluation


In [ ]:
y_prediction_knn = knn.predict(X_test)
y_prediction_knn


In [ ]:
print("Confusion Matrix")
print(confusion_matrix(y_test, y_prediction_knn))

print("\nClassification Report")
print(classification_report(y_test, y_prediction_knn))


## 14. Model 3: Gaussian Naive Bayes Classifier

Naive Bayes applies Bayes' theorem under a "naive" independence assumption between features, with a Gaussian (normal) likelihood per feature per class. It trains instantly and often performs well on small clinical datasets, even when the independence assumption is only approximately true.


In [ ]:
from sklearn.naive_bayes import GaussianNB

model_gaussian_naive_bayes = GaussianNB()
model_gaussian_naive_bayes.fit(X_train_resampled, y_train_resampled)


### 14.1 Model Predictions and Evaluation


In [ ]:
y_predict_gaussian_naive_bayes = model_gaussian_naive_bayes.predict(X_test)
print(y_predict_gaussian_naive_bayes)


In [ ]:
print("Confusion Matrix")
print(confusion_matrix(y_test, y_predict_gaussian_naive_bayes))

print("\nClassification Report")
print(classification_report(y_test, y_predict_gaussian_naive_bayes))

print("\nAccuracy:")
print(accuracy_score(y_test, y_predict_gaussian_naive_bayes))


## 15. Summary and Next Steps

### What this notebook does

1. Loads the 768-record diabetes dataset and confirms there are no `NaN`s, but identifies physiologically impossible zeros as hidden missing values.
2. Imputes those zeros with the mean (symmetric features) or the median (skewed features).
3. Removes outliers with the IQR rule, then trims the remaining Insulin tail at the 95th percentile.
4. Standardises all features to mean 0 / standard deviation 1.
5. Splits into train (67%) and test (33%) sets and balances the training data with SMOTE.
6. Trains and evaluates Logistic Regression, KNN and Gaussian Naive Bayes, and persists the Logistic Regression model with pickle.

### Comparing the models

Judge the three classifiers primarily on the **recall of the `Diabetic` class** in their classification reports, not on accuracy alone, given the healthcare cost of false negatives.

### Production caveats and next steps

- **Data leakage:** the scaler is fitted on the full dataset before the split (as in the original notebook). In a strict production setting, wrap imputation, scaling and the model in a `sklearn.pipeline.Pipeline` so that all preprocessing statistics are learned from the training folds only.
- **Stratification:** use `train_test_split(..., stratify=y)` to preserve the class ratio in both splits.
- **Cross-validation:** replace the single hold-out split with k-fold cross-validation for more stable estimates.
- **Tuning:** search hyperparameters (e.g. `n_neighbors` for KNN, `C` for Logistic Regression) with `GridSearchCV`.
- **Calibration and thresholding:** for clinical use, calibrate predicted probabilities and choose the decision threshold from the recall/precision trade-off rather than using the default 0.5.
